# Velocity model -> ASAGI NetCDF on Google Colab

Converts the raw CVM velocity slices (`cvm_s4.26.m01/`) into the ASAGI
material NetCDF that SeisSol reads, by running the project's
`generate_velocity_nc_from_raw.py` and reporting its built-in self-check.

### Before you start - put the inputs on Google Drive
Upload these next to each other on Drive (keeps the layout simple):
```
<your_case>/cvm_s4.26.m01/                      (the 38 velocity_raw_*.bp slices)
<your_case>/generate_velocity_nc_from_raw.py    (the converter; a sub-folder of
                                                 the same name also works)
```
Then set `RAW_DIR` in step 3 to the `cvm_s4.26.m01/` folder.

Output `safs_material_cvm_s4.26.m01.nc` (~112 MB) is written next to the
slices. We **import** the converter and call its `main()` (no `!python`), so
the wrong-`python3`-on-PATH pitfall can't happen and we can read its exit code.

## What it does (background)

A SeisSol run needs the rock's elastic moduli everywhere: density `rho`
[kg/m^3] and the Lame parameters `mu`, `lambda` [Pa]. The published model gives
seismic wave speeds instead; elasticity links them (plain-text math):
```
mu     = rho * Vs^2
lambda = rho * (Vp^2 - 2*Vs^2)      # needs Vp > sqrt(2)*Vs, else lambda <= 0
```
The converter's pipeline:
- **Stage 1:** read each `velocity_raw_*.bp` depth slice; reproject lon/lat
  (EPSG:4326) -> UTM 11N metres (EPSG:32611) with `pyproj`; resample onto a
  regular UTM grid (`scipy` LinearNDInterpolator; out-of-hull cells fail loud).
- **Stage 2:** `Vp,Vs,rho -> rho,mu,lambda`; resample onto a uniform `z` axis
  (ASAGI needs equidistant axes); write COARDS NetCDF4; run a 1000-point,
  fixed-seed self-check -> PASS / FAIL.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies
Colab already has **numpy** and **scipy**; it lacks **pyproj** (map
projection, EPSG:4326 -> EPSG:32611) and **netCDF4** (writes the ASAGI file).
We install just those.

In [ ]:
!pip -q install pyproj netCDF4
import numpy, scipy, pyproj, netCDF4
print("pyproj", pyproj.__version__, "| netCDF4", netCDF4.__version__,
      "| scipy", scipy.__version__, "| numpy", numpy.__version__)

## 3. Point at your data on Drive
**Edit `RAW_DIR`** to the folder that holds the `velocity_raw_*.bp` slices
(it always begins with `/content/drive/MyDrive/`). Not sure of the path? Run
`!find /content/drive/MyDrive -type d -name 'cvm_s4.26.m01'` in a scratch cell.

The output path and the converter are derived from `RAW_DIR` and checked with
asserts, so a wrong path fails here with a clear message (not deep in Stage 1).

In [ ]:
import os, glob

RAW_DIR = "/content/drive/MyDrive/safs_seisol_v2_2_0_LSW_s4.26.m01cvm/cvm_s4.26.m01"  # <-- EDIT

case_dir = os.path.dirname(RAW_DIR)                 # folder holding cvm_s4.26.m01/
OUT_NC   = os.path.join(case_dir, "safs_material_cvm_s4.26.m01.nc")

# auto-find the converter: a sibling sub-folder, or directly beside the data
_cands = [os.path.join(case_dir, "generate_velocity_nc_from_raw",
                       "generate_velocity_nc_from_raw.py"),
          os.path.join(case_dir, "generate_velocity_nc_from_raw.py")]
CONVERTER = next((c for c in _cands if os.path.exists(c)), "")

slices = sorted(glob.glob(os.path.join(RAW_DIR, "velocity_raw_*.bp")))
assert os.path.isdir(RAW_DIR), f"{RAW_DIR} is not a folder - fix RAW_DIR"
assert slices, (f"no velocity_raw_*.bp in {RAW_DIR} - RAW_DIR must point AT "
                "cvm_s4.26.m01, not its parent")
assert CONVERTER, ("generate_velocity_nc_from_raw.py not found near "
                   f"{case_dir} - upload it there or set CONVERTER")
print("converter :", CONVERTER)
print("raw slices:", RAW_DIR, f"({len(slices)} slices; expect 38)")
print("output    :", OUT_NC)

## 4. Run the converter
We import `generate_velocity_nc_from_raw.py` and call its `main()` in-process.
`--raw-dir` is passed explicitly (= your `RAW_DIR`); the converter's own
default points elsewhere.

**`DZ` knob:** `None` = production `dz = 250 m`, which prints a `WARNING` that
4 shallow source levels (-100..-400 m) fall off the grid -- **expected**, it
smooths the weathered top ~400 m. Set `DZ = 50` to keep every level (~5x
larger file). Output streams live; `main()` returns `0` on self-check PASS.

In [ ]:
import importlib.util, sys

GRID_DX, DZ, DTYPE = None, None, None   # None = converter's production default

spec = importlib.util.spec_from_file_location("gen_vel_nc", CONVERTER)
mod  = importlib.util.module_from_spec(spec); sys.modules[spec.name] = mod
spec.loader.exec_module(mod)

argv = ["--raw-dir", RAW_DIR, "--out", OUT_NC]
if GRID_DX is not None: argv += ["--grid-dx", str(GRID_DX)]
if DZ      is not None: argv += ["--dz",      str(DZ)]
if DTYPE   is not None: argv += ["--dtype",   str(DTYPE)]
print("running main(", argv, ")\n", flush=True)

rc = mod.main(argv)                     # 0 = self-check PASS, 1 = FAIL
print("\nreturn code:", rc, "->", "PASS" if rc == 0 else "FAIL")
assert rc == 0, "self-check FAILED - read the converter output above"
print("saved:", OUT_NC, f"({os.path.getsize(OUT_NC)/1e6:.1f} MB, on your Drive)")

## 5. Inspect the result (value ranges)
Reopen the NetCDF and back out `Vs = sqrt(mu/rho)`, `Vp = sqrt((lambda+2mu)/rho)`
to sanity-check that the stored moduli match the input model's speed ranges.

In [ ]:
import netCDF4, numpy as np

ds = netCDF4.Dataset(OUT_NC, "r")
x, y, z = ds["x"][:], ds["y"][:], ds["z"][:]
d = ds["data"][:]                       # compound array, shape (z, y, x)
rho = np.asarray(d["rho"]); mu = np.asarray(d["mu"]); lam = np.asarray(d["lambda"])
ds.close()

Vs = np.sqrt(mu / rho); Vp = np.sqrt((lam + 2*mu) / rho)
print(f"grid : x{len(x)} y{len(y)} z{len(z)}  "
      f"(z {z.min():.0f}..{z.max():.0f} m, dz {np.diff(z)[0]:.0f} m)")
print(f"rho  : {rho.min():8.0f} .. {rho.max():8.0f}  kg/m^3")
print(f"mu   : {mu.min():.3g} .. {mu.max():.3g}  Pa")
print(f"Vs   : {Vs.min():8.0f} .. {Vs.max():8.0f}  m/s")
print(f"Vp   : {Vp.min():8.0f} .. {Vp.max():8.0f}  m/s")

## 6. Plot Vs (surface map + depth profile)
A quick visual check: the shear-wave speed across the model at the surface
(top `z` level), and its variation with depth at the model centre.

In [ ]:
import matplotlib.pyplot as plt

Vs_surf = Vs[-1, :, :]                  # top elevation -> (y, x)
jc, ic  = len(y)//2, len(x)//2          # model-centre column

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
pc = ax[0].pcolormesh(x/1e3, y/1e3, Vs_surf, cmap="viridis", shading="auto")
ax[0].set_aspect("equal"); ax[0].set_title("Vs at surface  [m/s]")
ax[0].set_xlabel("UTM x [km]"); ax[0].set_ylabel("UTM y [km]")
fig.colorbar(pc, ax=ax[0], shrink=0.85)

ax[1].plot(Vs[:, jc, ic], z/1e3)
ax[1].set_xlabel("Vs [m/s]"); ax[1].set_ylabel("elevation z [km]")
ax[1].set_title("Vs depth profile (model centre)"); ax[1].grid(True)
plt.tight_layout(); plt.show()

## Notes / caveats
- The converter's **self-check** (step 4) is the real validation: it resamples
  1000 fixed-seed points two independent ways. `median rel err ~1e-8` + PASS
  means the NetCDF faithfully encodes the source moduli.
- At `dz = 250` the `WARNING` about 4 dropped shallow levels (-100..-400 m) is
  **expected** (deliberate top-400 m smoothing). Use `DZ = 50` (step 4) to keep
  them, at ~5x the file size.
- Inputs/outputs are large (~26 MB in, ~112 MB out) -> keep them on Drive so
  they persist; re-running step 4 overwrites the `.nc`.
- To convert another model, point `RAW_DIR` at its slice folder and re-run
  steps 3-6.